# Notebook 2: Model Training

**Purpose:** Train all 11 deep learning models for DR classification on EyePACS.

**Models:**
- DenseNet121 (binary + 5-class)
- RETFound (binary + 5-class + adversarial)
- EfficientNet-B3 (binary + 5-class)
- ResNet50 (binary + 5-class)
- ViT-Base (binary + 5-class)

**Requirements:** Google Colab Pro with T4 GPU, ~48 GPU-hours total.

**Output:** Trained model weights saved to Google Drive (`models_eyepacs_full/`).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm transformers

# Verify GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Part 1: Train DenseNet121, RETFound (5 models)

Trains DenseNet121 (binary + 5-class), RETFound (binary + 5-class + adversarial).
Models save to Drive as they complete — disconnect-safe.

In [ ]:
# ============================================================
# FULL EXPERIMENT: Train all 5 models on expanded EyePACS
# Models save to Drive as they complete — disconnect-safe
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from itertools import combinations
import json, os, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Paths
drive_base = '/content/drive/MyDrive/stochastic_orthogonality'
FULL_DIR = f'{drive_base}/data/eyepacs_full'
PROCESSED_DIR = f'{FULL_DIR}/processed'
MODEL_DIR = f'{drive_base}/models_eyepacs_full'
RESULTS_DIR = f'{drive_base}/results_eyepacs_full'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ---- Dataset classes ----

class DRDatasetBinary(Dataset):
    def __init__(self, csv_path, image_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        existing = set(os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.endswith('.png'))
        self.df = self.df[self.df['image_id'].isin(existing)].reset_index(drop=True)
        print(f'  Binary dataset: {len(self.df)} images | Referable: {self.df["binary_label"].mean():.1%}')
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, f"{row['image_id']}.png")).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(row['binary_label'], dtype=torch.float32), row['image_id']

class DRDataset5Class(Dataset):
    def __init__(self, csv_path, image_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        existing = set(os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.endswith('.png'))
        self.df = self.df[self.df['image_id'].isin(existing)].reset_index(drop=True)
        print(f'  5-class dataset: {len(self.df)} images | Grades: {self.df["level"].value_counts().sort_index().to_dict()}')
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, f"{row['image_id']}.png")).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(row['level'], dtype=torch.long), row['image_id'], torch.tensor(row['binary_label'], dtype=torch.float32)

class DRDatasetAdversarial(Dataset):
    def __init__(self, csv_path, image_dir, transform, primary_probs):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        self.primary_probs = primary_probs
        existing = set(os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.endswith('.png'))
        self.df = self.df[self.df['image_id'].isin(existing)].reset_index(drop=True)
        self.df = self.df[self.df['image_id'].isin(primary_probs.keys())].reset_index(drop=True)
        print(f'  Adversarial dataset: {len(self.df)} images with primary predictions')
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, f"{row['image_id']}.png")).convert('RGB')
        if self.transform: img = self.transform(img)
        label = torch.tensor(row['binary_label'], dtype=torch.float32)
        primary_p = torch.tensor(self.primary_probs[row['image_id']], dtype=torch.float32)
        return img, label, row['image_id'], primary_p

# ---- Helper functions ----

def build_model(arch_name, pretrained=True):
    return timm.create_model(arch_name, pretrained=pretrained, num_classes=1).to(device)

def get_predictions_binary(model, data_loader):
    model.eval()
    probs, labs, ids = [], [], []
    with torch.no_grad():
        for images, labels, img_ids in data_loader:
            images = images.to(device)
            outputs = torch.sigmoid(model(images).squeeze(-1))
            probs.extend(outputs.cpu().numpy())
            labs.extend(labels.numpy())
            ids.extend(img_ids)
    return np.array(probs), np.array(labs), ids

# ---- Load datasets ----
print('Loading datasets...')
train_bin = DRDatasetBinary(f'{FULL_DIR}/eyepacs_full_train.csv', PROCESSED_DIR, train_transform)
val_bin = DRDatasetBinary(f'{FULL_DIR}/eyepacs_full_val.csv', PROCESSED_DIR, val_transform)
test_bin = DRDatasetBinary(f'{FULL_DIR}/eyepacs_full_test.csv', PROCESSED_DIR, val_transform)

train_5c = DRDataset5Class(f'{FULL_DIR}/eyepacs_full_train.csv', PROCESSED_DIR, train_transform)
val_5c = DRDataset5Class(f'{FULL_DIR}/eyepacs_full_val.csv', PROCESSED_DIR, val_transform)

val_loader_bin = DataLoader(val_bin, batch_size=32, shuffle=False, num_workers=2)
test_loader_bin = DataLoader(test_bin, batch_size=32, shuffle=False, num_workers=2)
train_loader_bin = DataLoader(train_bin, batch_size=32, shuffle=True, num_workers=2)

train_loader_5c = DataLoader(train_5c, batch_size=8, shuffle=True, num_workers=2)
val_loader_5c = DataLoader(val_5c, batch_size=8, shuffle=False, num_workers=2)

# ============================================================
# MODEL 1: DenseNet121 Binary
# ============================================================
model_path = f'{MODEL_DIR}/densenet121_binary.pt'
if os.path.exists(model_path):
    print(f'\n{"="*60}')
    print('MODEL 1: DenseNet121 Binary — FOUND, skipping')
    print(f'{"="*60}')
else:
    print(f'\n{"="*60}')
    print('MODEL 1: DenseNet121 Binary (15 epochs)')
    print(f'{"="*60}')

    model1 = build_model('densenet121', pretrained=True)
    optimizer1 = optim.AdamW(model1.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler1 = optim.lr_scheduler.CosineAnnealingLR(optimizer1, T_max=15)
    criterion1 = nn.BCEWithLogitsLoss()
    best_auc1, best_state1 = 0, None

    for epoch in range(15):
        model1.train()
        tloss, nb = 0, 0
        for images, labels, _ in train_loader_bin:
            images, labels = images.to(device), labels.to(device)
            optimizer1.zero_grad()
            loss = criterion1(model1(images).squeeze(-1), labels)
            loss.backward()
            optimizer1.step()
            tloss += loss.item(); nb += 1
        scheduler1.step()

        model1.eval()
        vp, vl = [], []
        with torch.no_grad():
            for images, labels, _ in val_loader_bin:
                images = images.to(device)
                vp.extend(torch.sigmoid(model1(images).squeeze(-1)).cpu().numpy())
                vl.extend(labels.numpy())
        vauc = roc_auc_score(vl, vp)
        if epoch % 3 == 0 or epoch == 14:
            print(f'  Epoch {epoch+1}/15 — loss: {tloss/nb:.4f}, val AUC: {vauc:.4f}')
        if vauc > best_auc1:
            best_auc1 = vauc
            best_state1 = {k: v.cpu().clone() for k, v in model1.state_dict().items()}

    torch.save(best_state1, model_path)
    print(f'  Saved. Best val AUC: {best_auc1:.4f}')
    del model1; torch.cuda.empty_cache()

# ============================================================
# MODEL 2: DenseNet121 5-Class
# ============================================================
model_path = f'{MODEL_DIR}/densenet121_5class.pt'
if os.path.exists(model_path):
    print(f'\n{"="*60}')
    print('MODEL 2: DenseNet121 5-Class — FOUND, skipping')
    print(f'{"="*60}')
else:
    print(f'\n{"="*60}')
    print('MODEL 2: DenseNet121 5-Class (15 epochs)')
    print(f'{"="*60}')

    model2 = timm.create_model('densenet121', pretrained=True, num_classes=5).to(device)
    optimizer2 = optim.AdamW(model2.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=15)
    criterion2 = nn.CrossEntropyLoss()
    best_auc2, best_state2 = 0, None

    for epoch in range(15):
        model2.train()
        tloss, nb = 0, 0
        for images, labels_5c, _, _ in train_loader_5c:
            images, labels_5c = images.to(device), labels_5c.to(device)
            optimizer2.zero_grad()
            loss = criterion2(model2(images), labels_5c)
            loss.backward()
            optimizer2.step()
            tloss += loss.item(); nb += 1
        scheduler2.step()

        model2.eval()
        vp, vl = [], []
        with torch.no_grad():
            for images, _, _, binary_labels in val_loader_5c:
                images = images.to(device)
                probs_5c = torch.softmax(model2(images), dim=-1)
                ref_prob = probs_5c[:, 2] + probs_5c[:, 3] + probs_5c[:, 4]
                vp.extend(ref_prob.cpu().numpy())
                vl.extend(binary_labels.numpy())
        vauc = roc_auc_score(vl, vp)
        if epoch % 3 == 0 or epoch == 14:
            print(f'  Epoch {epoch+1}/15 — loss: {tloss/nb:.4f}, binary AUC: {vauc:.4f}')
        if vauc > best_auc2:
            best_auc2 = vauc
            best_state2 = {k: v.cpu().clone() for k, v in model2.state_dict().items()}

    torch.save(best_state2, model_path)
    print(f'  Saved. Best val binary AUC: {best_auc2:.4f}')
    del model2; torch.cuda.empty_cache()

# ============================================================
# MODEL 3: RETFound Binary
# ============================================================
model_path = f'{MODEL_DIR}/retfound_binary.pt'
if os.path.exists(model_path):
    print(f'\n{"="*60}')
    print('MODEL 3: RETFound Binary — FOUND, skipping')
    print(f'{"="*60}')
else:
    print(f'\n{"="*60}')
    print('MODEL 3: RETFound Binary (8 epochs)')
    print(f'{"="*60}')

    from transformers import AutoModel

    class RETFoundBinary(nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base = base
            self.classifier = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, 1))
        def forward(self, x):
            return self.classifier(self.base(x).last_hidden_state[:, 0])

    model3 = RETFoundBinary(AutoModel.from_pretrained("iszt/RETFound_mae_meh")).to(device)

    pg3 = []
    lr3 = 5e-5
    for i, layer in enumerate(model3.base.encoder.layer):
        decay = 0.65 ** (len(model3.base.encoder.layer) - i)
        pg3.append({'params': layer.parameters(), 'lr': lr3 * decay})
    pg3.append({'params': model3.base.embeddings.parameters(), 'lr': lr3 * 0.65**24})
    pg3.append({'params': model3.classifier.parameters(), 'lr': lr3 * 10})

    optimizer3 = optim.AdamW(pg3, weight_decay=0.05)
    scheduler3 = optim.lr_scheduler.CosineAnnealingLR(optimizer3, T_max=8)
    criterion3 = nn.BCEWithLogitsLoss()
    best_auc3, best_state3 = 0, None

    for epoch in range(8):
        model3.train()
        tloss, nb = 0, 0
        for images, labels, _ in train_loader_bin:
            images, labels = images.to(device), labels.to(device)
            optimizer3.zero_grad()
            loss = criterion3(model3(images).squeeze(-1), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model3.parameters(), 1.0)
            optimizer3.step()
            tloss += loss.item(); nb += 1
        scheduler3.step()

        model3.eval()
        vp, vl = [], []
        with torch.no_grad():
            for images, labels, _ in val_loader_bin:
                images = images.to(device)
                vp.extend(torch.sigmoid(model3(images).squeeze(-1)).cpu().numpy())
                vl.extend(labels.numpy())
        vauc = roc_auc_score(vl, vp)
        print(f'  Epoch {epoch+1}/8 — loss: {tloss/nb:.4f}, val AUC: {vauc:.4f}')
        if vauc > best_auc3:
            best_auc3 = vauc
            best_state3 = {k: v.cpu().clone() for k, v in model3.state_dict().items()}

    torch.save(best_state3, model_path)
    print(f'  Saved. Best val AUC: {best_auc3:.4f}')
    del model3; torch.cuda.empty_cache()

# ============================================================
# MODEL 4: RETFound 5-Class
# ============================================================
model_path = f'{MODEL_DIR}/retfound_5class.pt'
if os.path.exists(model_path):
    print(f'\n{"="*60}')
    print('MODEL 4: RETFound 5-Class — FOUND, skipping')
    print(f'{"="*60}')
else:
    print(f'\n{"="*60}')
    print('MODEL 4: RETFound 5-Class (10 epochs)')
    print(f'{"="*60}')

    from transformers import AutoModel

    class RETFound5Class(nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base = base
            self.classifier = nn.Sequential(
                nn.LayerNorm(1024), nn.Dropout(0.1), nn.Linear(1024, 5)
            )
        def forward(self, x):
            return self.classifier(self.base(x).last_hidden_state[:, 0])
        def predict_binary(self, x):
            probs = torch.softmax(self.forward(x), dim=-1)
            return probs[:, 2] + probs[:, 3] + probs[:, 4]

    model4 = RETFound5Class(AutoModel.from_pretrained("iszt/RETFound_mae_meh")).to(device)

    pg4 = []
    lr4 = 5e-5
    for i, layer in enumerate(model4.base.encoder.layer):
        decay = 0.65 ** (len(model4.base.encoder.layer) - i)
        pg4.append({'params': layer.parameters(), 'lr': lr4 * decay})
    pg4.append({'params': model4.base.embeddings.parameters(), 'lr': lr4 * 0.65**24})
    pg4.append({'params': model4.classifier.parameters(), 'lr': lr4 * 10})

    optimizer4 = optim.AdamW(pg4, weight_decay=0.05)
    scheduler4 = optim.lr_scheduler.CosineAnnealingLR(optimizer4, T_max=10)
    criterion4 = nn.CrossEntropyLoss()
    best_auc4, best_state4 = 0, None

    for epoch in range(10):
        model4.train()
        tloss, nb = 0, 0
        for images, labels_5c, _, _ in train_loader_5c:
            images, labels_5c = images.to(device), labels_5c.to(device)
            optimizer4.zero_grad()
            loss = criterion4(model4(images), labels_5c)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model4.parameters(), 1.0)
            optimizer4.step()
            tloss += loss.item(); nb += 1
        scheduler4.step()

        model4.eval()
        vp, vl = [], []
        with torch.no_grad():
            for images, _, _, binary_labels in val_loader_5c:
                images = images.to(device)
                ref_prob = model4.predict_binary(images)
                vp.extend(ref_prob.cpu().numpy())
                vl.extend(binary_labels.numpy())
        vauc = roc_auc_score(vl, vp)
        print(f'  Epoch {epoch+1}/10 — loss: {tloss/nb:.4f}, binary AUC: {vauc:.4f}')
        if vauc > best_auc4:
            best_auc4 = vauc
            best_state4 = {k: v.cpu().clone() for k, v in model4.state_dict().items()}

    torch.save(best_state4, model_path)
    print(f'  Saved. Best val binary AUC: {best_auc4:.4f}')
    del model4; torch.cuda.empty_cache()

# ============================================================
# MODEL 5: RETFound Adversarial (alpha=0.08)
# Needs primary model predictions first
# ============================================================
model_path = f'{MODEL_DIR}/retfound_adversarial.pt'
if os.path.exists(model_path):
    print(f'\n{"="*60}')
    print('MODEL 5: RETFound Adversarial — FOUND, skipping')
    print(f'{"="*60}')
else:
    print(f'\n{"="*60}')
    print('MODEL 5: RETFound Adversarial alpha=0.08 (8 epochs)')
    print(f'{"="*60}')

    # Load DenseNet121 binary as primary model
    print('  Loading primary model for adversarial training...')
    primary = build_model('densenet121', pretrained=False)
    primary.load_state_dict(torch.load(f'{MODEL_DIR}/densenet121_binary.pt', map_location=device))
    primary = primary.to(device)
    primary.eval()

    # Cache primary predictions on train and val
    primary_train_probs = {}
    with torch.no_grad():
        for images, labels, img_ids in train_loader_bin:
            images = images.to(device)
            probs = torch.sigmoid(primary(images).squeeze(-1))
            for img_id, p in zip(img_ids, probs.cpu()):
                primary_train_probs[img_id] = p.item()

    primary_val_probs = {}
    with torch.no_grad():
        for images, labels, img_ids in val_loader_bin:
            images = images.to(device)
            probs = torch.sigmoid(primary(images).squeeze(-1))
            for img_id, p in zip(img_ids, probs.cpu()):
                primary_val_probs[img_id] = p.item()
    del primary; torch.cuda.empty_cache()
    print(f'  Cached {len(primary_train_probs)} train, {len(primary_val_probs)} val predictions')

    # Adversarial dataset
    train_adv = DRDatasetAdversarial(
        f'{FULL_DIR}/eyepacs_full_train.csv', PROCESSED_DIR, train_transform, primary_train_probs
    )
    val_adv = DRDatasetAdversarial(
        f'{FULL_DIR}/eyepacs_full_val.csv', PROCESSED_DIR, val_transform, primary_val_probs
    )
    train_loader_adv = DataLoader(train_adv, batch_size=8, shuffle=True, num_workers=2)
    val_loader_adv = DataLoader(val_adv, batch_size=8, shuffle=False, num_workers=2)

    # Decorrelation loss
    class DecorrelationLoss(nn.Module):
        def __init__(self, alpha=0.08):
            super().__init__()
            self.bce = nn.BCEWithLogitsLoss(reduction='none')
            self.alpha = alpha
        def forward(self, logits, labels, primary_probs):
            cls_loss = self.bce(logits, labels).mean()
            probs = torch.sigmoid(logits)
            se = torch.abs(probs - labels)
            pe = torch.abs(primary_probs - labels)
            se_c = se - se.mean()
            pe_c = pe - pe.mean()
            corr = (se_c * pe_c).mean() / (se_c.std() * pe_c.std() + 1e-8)
            decorr = torch.clamp(corr, min=0.0)
            return cls_loss + self.alpha * decorr, cls_loss.item(), decorr.item()

    from transformers import AutoModel

    class RETFoundBinaryAdv(nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base = base
            self.classifier = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, 1))
        def forward(self, x):
            return self.classifier(self.base(x).last_hidden_state[:, 0])

    model5 = RETFoundBinaryAdv(AutoModel.from_pretrained("iszt/RETFound_mae_meh")).to(device)

    pg5 = []
    lr5 = 3e-5
    for i, layer in enumerate(model5.base.encoder.layer):
        decay = 0.65 ** (len(model5.base.encoder.layer) - i)
        pg5.append({'params': layer.parameters(), 'lr': lr5 * decay})
    pg5.append({'params': model5.base.embeddings.parameters(), 'lr': lr5 * 0.65**24})
    pg5.append({'params': model5.classifier.parameters(), 'lr': lr5 * 10})

    optimizer5 = optim.AdamW(pg5, weight_decay=0.05)
    scheduler5 = optim.lr_scheduler.CosineAnnealingLR(optimizer5, T_max=8)
    criterion5 = DecorrelationLoss(alpha=0.08)
    best_auc5, best_state5 = 0, None

    for epoch in range(8):
        model5.train()
        ec, ed, nb = 0, 0, 0
        for images, labels, _, primary_p in train_loader_adv:
            images, labels, primary_p = images.to(device), labels.to(device), primary_p.to(device)
            optimizer5.zero_grad()
            loss, cl, dl = criterion5(model5(images).squeeze(-1), labels, primary_p)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model5.parameters(), 1.0)
            optimizer5.step()
            ec += cl; ed += dl; nb += 1
        scheduler5.step()

        model5.eval()
        vp, vl = [], []
        with torch.no_grad():
            for images, labels, _, _ in val_loader_adv:
                images = images.to(device)
                vp.extend(torch.sigmoid(model5(images).squeeze(-1)).cpu().numpy())
                vl.extend(labels.numpy())
        vauc = roc_auc_score(vl, vp)
        print(f'  Epoch {epoch+1}/8 — cls: {ec/nb:.4f}, decorr: {ed/nb:.4f}, val AUC: {vauc:.4f}')
        if vauc > best_auc5:
            best_auc5 = vauc
            best_state5 = {k: v.cpu().clone() for k, v in model5.state_dict().items()}

    torch.save(best_state5, model_path)
    print(f'  Saved. Best val AUC: {best_auc5:.4f}')
    del model5; torch.cuda.empty_cache()

print(f'\n\n{"="*60}')
print('ALL 5 MODELS TRAINED AND SAVED TO DRIVE')
print(f'{"="*60}')
for f in os.listdir(MODEL_DIR):
    if f.endswith('.pt'):
        size = os.path.getsize(os.path.join(MODEL_DIR, f)) / 1e6
        print(f'  {f}: {size:.1f} MB')

## Part 2: Train EfficientNet-B3, ResNet50, ViT-Base (6 models)

Trains each architecture in binary and 5-class formulations.

In [ ]:
# ============================================================
# TRAIN 6 ADDITIONAL MODELS ON EYEPACS
# EfficientNet-B3, ResNet50, ViT-Base × Binary and 5-Class
# ============================================================

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
import timm
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from PIL import Image
import torchvision.transforms as transforms
import os, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

drive_base = '/content/drive/MyDrive/stochastic_orthogonality'
FULL_DIR = f'{drive_base}/data/eyepacs_full'
PROCESSED_DIR = f'{FULL_DIR}/processed'
MODEL_DIR = f'{drive_base}/models_eyepacs_full'

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class DRBinary(torch.utils.data.Dataset):
    def __init__(self, csv_path, image_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        existing = set(os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.endswith('.png'))
        self.df = self.df[self.df['image_id'].isin(existing)].reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, f"{row['image_id']}.png")).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(row['binary_label'], dtype=torch.float32), row['image_id']

class DR5Class(torch.utils.data.Dataset):
    def __init__(self, csv_path, image_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        existing = set(os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.endswith('.png'))
        self.df = self.df[self.df['image_id'].isin(existing)].reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, f"{row['image_id']}.png")).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(row['level'], dtype=torch.long), row['image_id'], torch.tensor(row['binary_label'], dtype=torch.float32)

# Load datasets
train_bin = DRBinary(f'{FULL_DIR}/eyepacs_full_train.csv', PROCESSED_DIR, train_transform)
val_bin = DRBinary(f'{FULL_DIR}/eyepacs_full_val.csv', PROCESSED_DIR, val_transform)
train_5c = DR5Class(f'{FULL_DIR}/eyepacs_full_train.csv', PROCESSED_DIR, train_transform)
val_5c = DR5Class(f'{FULL_DIR}/eyepacs_full_val.csv', PROCESSED_DIR, val_transform)

train_loader_bin = DataLoader(train_bin, batch_size=32, shuffle=True, num_workers=2)
val_loader_bin = DataLoader(val_bin, batch_size=32, shuffle=False, num_workers=2)
train_loader_5c = DataLoader(train_5c, batch_size=32, shuffle=True, num_workers=2)
val_loader_5c = DataLoader(val_5c, batch_size=32, shuffle=False, num_workers=2)

print(f'Train: {len(train_bin)}, Val: {len(val_bin)}')

# Models to train
MODELS_TO_TRAIN = [
    ('efficientnet_b3', 'binary', 'efficientnet_b3', 15),
    ('efficientnet_b3', '5class', 'efficientnet_b3', 15),
    ('resnet50', 'binary', 'resnet50', 15),
    ('resnet50', '5class', 'resnet50', 15),
    ('vit_base', 'binary', 'vit_base_patch16_224', 15),
    ('vit_base', '5class', 'vit_base_patch16_224', 15),
]

for arch_name, task, timm_name, n_epochs in MODELS_TO_TRAIN:
    save_name = f'{arch_name}_{task}'
    model_path = f'{MODEL_DIR}/{save_name}.pt'

    if os.path.exists(model_path):
        print(f'\n{"="*60}')
        print(f'{save_name} — FOUND, skipping')
        print(f'{"="*60}')
        continue

    print(f'\n{"="*60}')
    print(f'{save_name} ({n_epochs} epochs)')
    print(f'{"="*60}')

    if task == 'binary':
        model = timm.create_model(timm_name, pretrained=True, num_classes=1).to(device)
        criterion = nn.BCEWithLogitsLoss()
        train_loader = train_loader_bin
        val_loader = val_loader_bin
    else:
        model = timm.create_model(timm_name, pretrained=True, num_classes=5).to(device)
        criterion = nn.CrossEntropyLoss()
        train_loader = train_loader_5c
        val_loader = val_loader_5c

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    best_auc, best_state = 0, None

    for epoch in range(n_epochs):
        model.train()
        tloss, nb = 0, 0

        if task == 'binary':
            for images, labels, _ in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(images).squeeze(-1), labels)
                loss.backward()
                optimizer.step()
                tloss += loss.item(); nb += 1
        else:
            for images, labels_5c, _, _ in train_loader:
                images, labels_5c = images.to(device), labels_5c.to(device)
                optimizer.zero_grad()
                loss = criterion(model(images), labels_5c)
                loss.backward()
                optimizer.step()
                tloss += loss.item(); nb += 1

        scheduler.step()

        # Validate — always measure binary AUC
        model.eval()
        vp, vl = [], []
        with torch.no_grad():
            if task == 'binary':
                for images, labels, _ in val_loader:
                    images = images.to(device)
                    vp.extend(torch.sigmoid(model(images).squeeze(-1)).cpu().numpy())
                    vl.extend(labels.numpy())
            else:
                for images, _, _, binary_labels in val_loader:
                    images = images.to(device)
                    probs_5c = torch.softmax(model(images), dim=-1)
                    ref_prob = probs_5c[:, 2] + probs_5c[:, 3] + probs_5c[:, 4]
                    vp.extend(ref_prob.cpu().numpy())
                    vl.extend(binary_labels.numpy())

        vauc = roc_auc_score(vl, vp)
        if epoch % 3 == 0 or epoch == n_epochs - 1:
            print(f'  Epoch {epoch+1}/{n_epochs} — loss: {tloss/nb:.4f}, val AUC: {vauc:.4f}')

        if vauc > best_auc:
            best_auc = vauc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    torch.save(best_state, model_path)
    print(f'  Saved. Best val AUC: {best_auc:.4f}')
    del model; torch.cuda.empty_cache()

print(f'\n\n{"="*60}')
print('ALL ADDITIONAL MODELS TRAINED')
print(f'{"="*60}')
for f in sorted(os.listdir(MODEL_DIR)):
    if f.endswith('.pt'):
        size = os.path.getsize(os.path.join(MODEL_DIR, f)) / 1e6
        print(f'  {f}: {size:.1f} MB')

## Verify All Models

In [ ]:
import os
MODEL_DIR = '/content/drive/MyDrive/stochastic_orthogonality/models_eyepacs_full'
print('Trained models:')
for f in sorted(os.listdir(MODEL_DIR)):
    if f.endswith('.pt'):
        size = os.path.getsize(os.path.join(MODEL_DIR, f)) / 1e6
        print(f'  {f}: {size:.1f} MB')
print(f'\nTotal: {len([f for f in os.listdir(MODEL_DIR) if f.endswith(".pt")])} models')
print('Ready for Notebook 3 (evaluation).')